# Jobicy data collection and cleaning

This notebook collects U.S.-eligible remote tech job postings from the Jobicy API and cleans them into the CSV files that are loaded into MySQL.

- Every API response is saved unchanged in a dated folder under `data/jobicy/raw/`, so later steps can always be rerun.
- Cleaning always reads those saved files, never the live API, so every cleaned row traces back to a saved response.
- Jobicy asks API users to credit Jobicy and link each posting to its original page, so `job_url` is kept.

Run the cells from top to bottom. Set `COLLECT_NEW_DATA = 1` only when you want a new pull from the API.

Install packages (first run only)

In [1]:
%pip install beautifulsoup4 requests

Note: you may need to restart the kernel to use updated packages.


Settings

In [1]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

# 1 = call the Jobicy API now and save a new dated raw folder
# 0 = skip collection and only clean the raw files already saved
COLLECT_NEW_DATA = 0

# 1 = write jobicy_all_jobs.csv and the cleaned CSV files
SAVE_FILES = 1

# Paths are relative to this notebook (notebooks/), so it runs on any computer
DATA_FOLDER = Path("..") / "data"
RAW_FOLDER = DATA_FOLDER / "jobicy" / "raw"
FLAT_RAW_FILE = DATA_FOLDER / "jobicy" / "jobicy_all_jobs.csv"
CLEANED_FOLDER = DATA_FOLDER / "cleaned"

# Jobicy API
url = "https://jobicy.com/api/jobs"

job_roles = [
    "Data Analyst",
    "Data Scientist",
    "Data Engineer",
    "Software Engineer",
    "AI Engineer"
]

print("Raw folder found:", RAW_FOLDER.exists())

Raw folder found: True


Step 0: Collect new postings from the Jobicy API

Runs only when `COLLECT_NEW_DATA = 1`. Each API response is saved unchanged in a new folder named by the UTC time of the pull, for example `data/jobicy/raw/2026-11-05T1500Z/data_analyst.json`. Earlier pulls are never overwritten, so postings build up over time.

Jobicy asks that a new pull start no more than once per hour. If a response says `hasMore`, the next page is requested with `nextCursor` and saved as `<role>_page2.json`, and so on.

In [2]:
if COLLECT_NEW_DATA == 1:

    # One folder per pull, named by the UTC time the pull started
    pull_time = datetime.now(timezone.utc)
    pull_folder = RAW_FOLDER / pull_time.strftime("%Y-%m-%dT%H%MZ")
    pull_folder.mkdir(parents=True, exist_ok=True)

    for role in job_roles:

        print("\nCollecting:", role)

        filename = role.lower().replace(" ", "_")
        cursor = None
        page = 1

        while True:

            params = {
                "geo": "usa",
                "tag": role,
                "count": 100
            }

            # Later pages are requested with the cursor from the previous response
            if cursor is not None:
                params["cursor"] = cursor

            try:
                response = requests.get(url, params=params, timeout=30)
                response.raise_for_status()
                data = response.json()
            except requests.RequestException as e:
                print("Request failed:", e)
                break

            if data.get("success") is False:
                print("API Error:", data.get("error"))
                break

            # Save the response exactly as received
            if page == 1:
                out_file = pull_folder / f"{filename}.json"
            else:
                out_file = pull_folder / f"{filename}_page{page}.json"

            with open(out_file, "w", encoding="utf-8") as file:
                json.dump(data, file, indent=2)

            print("Page", page, "jobs found:", len(data.get("jobs", [])))

            if data.get("hasMore") and data.get("nextCursor"):
                cursor = data["nextCursor"]
                page = page + 1
                time.sleep(1)
            else:
                break

        time.sleep(1)

    print("\nSaved new pull in folder:", pull_folder.name)

else:
    print("Skipped collection (COLLECT_NEW_DATA = 0). Using the raw files already saved.")

Skipped collection (COLLECT_NEW_DATA = 0). Using the raw files already saved.


Step 1: Load every saved raw response

Reads every pull folder under `data/jobicy/raw/`. Each record gets the search tag that returned it (`searched_role`, read from the saved response) and the time of the pull (`collected_at`, read from the folder name).

The flattened file `data/jobicy/jobicy_all_jobs.csv` keeps one row per posting per search per pull. Duplicates are removed in Step 2, not here.

In [3]:
raw_records = []

pull_folders = sorted(p for p in RAW_FOLDER.iterdir() if p.is_dir())

for pull_folder in pull_folders:

    # Folder name is the UTC time of the pull, e.g. 2026-10-31T1852Z
    collected_at = datetime.strptime(pull_folder.name, "%Y-%m-%dT%H%MZ").replace(tzinfo=timezone.utc)

    records_before = len(raw_records)
    raw_files = sorted(pull_folder.glob("*.json"))

    for raw_file in raw_files:

        with open(raw_file, encoding="utf-8") as file:
            data = json.load(file)

        # The search tag is stored in the response itself
        searched_role = data.get("appliedFilters", {}).get("tag")

        for job in data.get("jobs", []):
            record = job.copy()
            record["searched_role"] = searched_role
            record["collected_at"] = collected_at
            raw_records.append(record)

    print(pull_folder.name, ":", len(raw_files), "files,", len(raw_records) - records_before, "records")

raw_jobs = pd.DataFrame(raw_records)

if raw_jobs.empty:
    raise ValueError("No raw Jobicy files found. Set COLLECT_NEW_DATA = 1 and run Step 0.")

if SAVE_FILES == 1:
    raw_jobs.to_csv(FLAT_RAW_FILE, index=False)

print("\nRaw records (posting x search x pull):", len(raw_jobs))
print("Unique Jobicy job IDs:", raw_jobs["id"].nunique())
print("\nRecords per search tag:")
print(raw_jobs["searched_role"].value_counts())

2026-10-31T1852Z : 5 files, 85 records

Raw records (posting x search x pull): 85
Unique Jobicy job IDs: 74

Records per search tag:
searched_role
Software Engineer    44
AI Engineer          15
Data Engineer        13
Data Scientist        8
Data Analyst          5
Name: count, dtype: int64


Step 2: Clean the postings

- One row per Jobicy job ID. When a posting comes back from several searches or pulls, the most recent copy is kept. `collected_at` is the first time it was seen and `last_seen_at` the last time.
- `searched_role` is the first search tag that returned the posting. It is not the role category, which comes from the job title in MySQL.
- HTML tags are removed from descriptions, and extra spaces are removed from titles, company names and locations.
- Kept for the dashboard: `job_url` (link to the original posting), the salary range, `job_level` and `job_industry`.

In [4]:
# Order of searches inside one pull, so "first search tag" is well defined
role_order = {}
for i, role in enumerate(job_roles):
    role_order[role] = i

jobs = raw_jobs.copy()
jobs["search_order"] = jobs["searched_role"].map(role_order)
jobs = jobs.sort_values(["collected_at", "search_order"])

# First and last time each posting was seen across all pulls
first_seen = jobs.groupby("id")["collected_at"].min()
last_seen = jobs.groupby("id")["collected_at"].max()

# First search tag that returned each posting
first_role = jobs.drop_duplicates(subset=["id"], keep="first").set_index("id")["searched_role"]

# Most recent copy of each posting (title or salary can change between pulls)
jobicy_clean = jobs.drop_duplicates(subset=["id"], keep="last").copy()

jobicy_clean["searched_role"] = jobicy_clean["id"].map(first_role)
jobicy_clean["collected_at"] = jobicy_clean["id"].map(first_seen)
jobicy_clean["last_seen_at"] = jobicy_clean["id"].map(last_seen)

# Salary fields are missing from a response when no posting in it lists pay
for column in ["salaryMin", "salaryMax", "salaryCurrency", "salaryPeriod"]:
    if column not in jobicy_clean.columns:
        jobicy_clean[column] = pd.NA

# Rename columns
jobicy_clean = jobicy_clean.rename(columns={
    "id": "job_id",
    "jobTitle": "job_title",
    "companyName": "company_name",
    "jobGeo": "location",
    "jobDescription": "description",
    "pubDate": "publication_date",
    "url": "job_url",
    "jobLevel": "job_level",
    "jobIndustry": "job_industry",
    "salaryMin": "salary_min",
    "salaryMax": "salary_max",
    "salaryCurrency": "salary_currency",
    "salaryPeriod": "salary_period"
})

# Remove HTML tags
def clean_html(text):

    if pd.isna(text):
        return ""

    soup = BeautifulSoup(str(text), "html.parser")

    return " ".join(soup.stripped_strings)

jobicy_clean["description_clean"] = (
    jobicy_clean["description"].apply(clean_html)
)

# Jobicy returns the industry as a list; so far every posting has exactly one
def industry_text(value):

    if isinstance(value, list):
        return "; ".join(value)

    if pd.isna(value):
        return ""

    return str(value)

jobicy_clean["job_industry"] = jobicy_clean["job_industry"].apply(industry_text)

# Remove extra spaces ("Canada,  USA" becomes "Canada, USA")
for column in ["job_title", "company_name", "location", "job_level", "job_url"]:
    jobicy_clean[column] = (
        jobicy_clean[column]
        .astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

# Salaries as numbers; postings without pay stay empty
jobicy_clean["salary_min"] = pd.to_numeric(jobicy_clean["salary_min"], errors="coerce")
jobicy_clean["salary_max"] = pd.to_numeric(jobicy_clean["salary_max"], errors="coerce")

# Convert publication dates
jobicy_clean["publication_date"] = (
    pd.to_datetime(
        jobicy_clean["publication_date"],
        errors="coerce",
        utc=True
    )
)

# Columns saved to jobicy_jobs_clean.csv. The first six keep their original order.
output_columns = [
    "job_id",
    "job_title",
    "company_name",
    "location",
    "description_clean",
    "publication_date",
    "job_url",
    "job_level",
    "job_industry",
    "salary_min",
    "salary_max",
    "salary_currency",
    "salary_period",
    "searched_role",
    "collected_at",
    "last_seen_at"
]

jobicy_clean = jobicy_clean[output_columns + ["description"]].reset_index(drop=True)

print("Total Unique Jobs:", len(jobicy_clean))
print("With a salary range:", jobicy_clean["salary_min"].notna().sum())
print("With a job URL:", jobicy_clean["job_url"].notna().sum())
print("\nJob level:")
print(jobicy_clean["job_level"].value_counts())

display(
    jobicy_clean[
        [
            "job_id",
            "job_title",
            "company_name",
            "location",
            "job_level",
            "salary_min",
            "salary_max"
        ]
    ].head()
)

Total Unique Jobs: 74
With a salary range: 54
With a job URL: 74

Job level:
job_level
Senior                 52
Director               11
Any                     8
Entry-Level, Junior     2
Midweight               1
Name: count, dtype: Int64


,job_id,job_title,company_name,location,job_level,salary_min,salary_max
0,147852,"Principal Analyst, Data Integration",H1,USA,Senior,145000.0,170000.0
1,158025,Senior Data Product Analyst,YipitData,USA,Senior,NaN,NaN
2,157762,Senior Data Analyst (MedTech),YipitData,USA,Senior,175000.0,200000.0
3,157759,Senior Data Analyst,YipitData,USA,Senior,170000.0,190000.0
4,156941,Senior Data Analyst - Marketing,Supabase,Anywhere,Senior,NaN,NaN


Step 3: Extract technical skills from Jobicy

This is still the starting dictionary of 10 skills.

In [5]:
import re

# Initial skill dictionary
skill_patterns = {
    "Python": r"\bpython\b",
    "SQL": r"\bsql\b",
    "Tableau": r"\btableau\b",
    "Power BI": r"\bpower\s*bi\b",
    "Microsoft Excel": r"\b(?:microsoft excel|ms excel)\b",
    "AWS": r"\b(?:aws|amazon web services)\b",
    "PostgreSQL": r"\b(?:postgresql|postgres)\b",
    "Snowflake": r"\bsnowflake\b",
    "Docker": r"\bdocker\b",
    "Java": r"\bjava\b"
}

# Extract skills from job descriptions
def extract_skills(description):

    found_skills = []

    for skill, pattern in skill_patterns.items():

        if re.search(
            pattern,
            description,
            flags=re.IGNORECASE
        ):
            found_skills.append(skill)

    return found_skills


jobicy_clean["extracted_skills"] = (
    jobicy_clean["description_clean"].apply(
        extract_skills
    )
)

display(
    jobicy_clean[
        [
            "job_title",
            "extracted_skills"
        ]
    ].head(10)
)

,job_title,extracted_skills
0,"Principal Analyst, Data Integration","[SQL, AWS]"
1,Senior Data Product Analyst,[SQL]
2,Senior Data Analyst (MedTech),[]
3,Senior Data Analyst,"[Python, SQL]"
4,Senior Data Analyst - Marketing,"[Python, SQL, PostgreSQL]"
5,"Data Scientist, Analytics (Technical Leadership)","[Python, SQL]"
6,Sr. Data Scientist,"[Python, SQL, AWS]"
7,"Senior Data Scientist, Product","[Python, SQL, AWS, PostgreSQL, Snowflake]"
8,"Data Scientist, Risk","[Python, SQL, AWS, Snowflake]"
9,"Senior Data Scientist, Full Stack","[Python, SQL]"


Step 4: Prepare the Job-Skills table

In [6]:
# Create job-skill relationships
job_skills_clean = jobicy_clean[
    [
        "job_id",
        "extracted_skills"
    ]
].explode("extracted_skills")

# Rename column
job_skills_clean = job_skills_clean.rename(
    columns={
        "extracted_skills": "skill_name"
    }
)

# Remove rows without extracted skills
job_skills_clean = job_skills_clean.dropna(
    subset=["skill_name"]
)

# Remove duplicate job-skill pairs
job_skills_clean = job_skills_clean.drop_duplicates(
    subset=["job_id", "skill_name"]
)

print(
    "Total Job-Skill Relationships:",
    len(job_skills_clean)
)

display(job_skills_clean.head(20))

Total Job-Skill Relationships: 167


,job_id,skill_name
0,147852,SQL
0,147852,AWS
1,158025,SQL
3,157759,Python
3,157759,SQL
4,156941,Python
4,156941,SQL
4,156941,PostgreSQL
5,141971,Python
5,141971,SQL


Step 5: Save the cleaned datasets

In [7]:
if SAVE_FILES == 1:

    CLEANED_FOLDER.mkdir(parents=True, exist_ok=True)

    # Store only scalar columns to avoid CSV list serialization issues
    jobicy_clean[output_columns].to_csv(
        CLEANED_FOLDER / "jobicy_jobs_clean.csv",
        index=False
    )

    # Save extracted job-skill relationships
    job_skills_clean.to_csv(
        CLEANED_FOLDER / "job_skills_clean.csv",
        index=False
    )

    print("Saved", len(jobicy_clean), "postings and", len(job_skills_clean), "job-skill links")

else:
    print("Not saved (SAVE_FILES = 0)")

Saved 74 postings and 167 job-skill links


Automating the jobicy data process using Python's MySQL connector.

In [6]:
import mysql.connector

connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Sheetal@24",
    database="jobavenir_db"
)

cursor = connection.cursor()

cursor.execute("SELECT COUNT(*) FROM occupations")

result = cursor.fetchone()

print("Total occupations:", result[0])

cursor.close()
connection.close()

Total occupations: 0
